# QA on Data Pipeline

This notebook validates whether incremental rolling snapshots reproduce the state that would be obtained from a fresh full source pull over the same analytical period.
The primary risks are:
1. records added after their event date,
2. records modified after they fall outside the incremental overlap window,
3. records deleted from the upstream dataset,
4. field-level corrections that do not affect row counts.

In [16]:
from pathlib import Path 
import pandas as pd

def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "dashboard").exists() and (candidate / "notebooks").exists():
            return candidate

    raise RuntimeError("Could not find repository root.")

REPO_ROOT = find_repo_root()

CALL_SNAPSHOT_DIR = REPO_ROOT / "data" / "processed"
CRIME_SNAPSHOT_DIR = REPO_ROOT / "data" / "processed" / "crime"

print("Repository root:", REPO_ROOT)
print("Calls snapshot:", CALL_SNAPSHOT_DIR)
print("Crime snapshot:", CRIME_SNAPSHOT_DIR)

Repository root: C:\Users\benca\code\PersonalPythonProjects\SPDCallDashboard
Calls snapshot: C:\Users\benca\code\PersonalPythonProjects\SPDCallDashboard\data\processed
Crime snapshot: C:\Users\benca\code\PersonalPythonProjects\SPDCallDashboard\data\processed\crime


In [17]:
CALL_ROLLING_DAYS = 734
CALL_OVERLAP_DAYS = 14

CRIME_ROLLING_DAYS = 734
CRIME_OVERLAP_DAYS = 30

CALL_KEY = "call_sign_dispatch_id"
CALL_EVENT_KEY = "cad_event_number"
CALL_TIME = "cad_event_original_time_queued"

CRIME_KEY = "offense_id"
CRIME_EVENT_TIME = "offense_date"
CRIME_REFRESH_TIME = "report_date_time"

from dashboard.spd_service import load_spd_call_dataset
from dashboard.spd_snapshot import load_spd_call_snapshot

from dashboard.crime_service import load_crime_dataset
from dashboard.crime_snapshot import load_crime_snapshot

rolling_calls, calls_metadata = load_spd_call_snapshot(
    CALL_SNAPSHOT_DIR
)

rolling_crime, crime_metadata = load_crime_snapshot(
    CRIME_SNAPSHOT_DIR
)

In [18]:
print(
    rolling_calls[CALL_TIME].min(),
    rolling_calls[CALL_TIME].max()
)

print(
    rolling_crime[CRIME_EVENT_TIME].min(),
    rolling_crime[CRIME_EVENT_TIME].max()
)

2024-09-15 00:04:00 2026-09-18 23:58:33
2024-09-17 23:00:00 2026-09-21 22:44:00


In [20]:
calls_start = (
    pd.to_datetime(rolling_calls[CALL_TIME]).min().date().isoformat()
)

full_calls = load_spd_call_dataset(
    start_date=calls_start,
    page_size=5000,
    max_pages=None,
    timeout=30,
    retry_backoff_seconds=30,
)

In [21]:
start = pd.to_datetime(rolling_calls[CALL_TIME]).min()
end = pd.to_datetime(rolling_calls[CALL_TIME]).max()

full_calls[CALL_TIME] = pd.to_datetime(full_calls[CALL_TIME])

full_calls_compare = full_calls[
    full_calls[CALL_TIME].between(start, end)
].copy()

rolling_calls_compare = rolling_calls[
    pd.to_datetime(rolling_calls[CALL_TIME]).between(start, end)
].copy()

In [23]:
crimes_start = (
    pd.to_datetime(rolling_calls[CALL_TIME]).min().date().isoformat()
)

full_crime = load_crime_dataset(
    start_date=crimes_start,
    page_size=5000,
    max_pages=None,
    timeout=30,
    retry_backoff_seconds=30,
)

Fetched page 1, records this page: 5000, total records: 5000
Fetched page 2, records this page: 5000, total records: 10000
Fetched page 3, records this page: 5000, total records: 15000
Fetched page 4, records this page: 5000, total records: 20000
Fetched page 5, records this page: 5000, total records: 25000
Fetched page 6, records this page: 5000, total records: 30000
Fetched page 7, records this page: 5000, total records: 35000
Fetched page 8, records this page: 5000, total records: 40000
Fetched page 9, records this page: 5000, total records: 45000
Fetched page 10, records this page: 5000, total records: 50000
Fetched page 11, records this page: 5000, total records: 55000
Fetched page 12, records this page: 5000, total records: 60000
Fetched page 13, records this page: 5000, total records: 65000
Fetched page 14, records this page: 5000, total records: 70000
Fetched page 15, records this page: 5000, total records: 75000
Fetched page 16, records this page: 5000, total records: 80000
Fe

In [45]:
full_crime[CRIME_EVENT_TIME] = pd.to_datetime(
    full_crime[CRIME_EVENT_TIME],
    errors="coerce",
)

full_crime[CRIME_REFRESH_TIME] = pd.to_datetime(
    full_crime[CRIME_REFRESH_TIME],
    errors="coerce",
)

In [47]:
start = pd.to_datetime(rolling_crime[CRIME_EVENT_TIME]).min()
end = pd.to_datetime(rolling_crime[CRIME_EVENT_TIME]).max()

full_crime[CRIME_EVENT_TIME] = pd.to_datetime(full_crime[CRIME_EVENT_TIME])

full_crime_compare = full_crime[
    full_crime[CRIME_EVENT_TIME].between(start, end)
].copy()

rolling_crime_compare = rolling_crime[
    pd.to_datetime(rolling_crime[CRIME_EVENT_TIME]).between(start, end)
].copy()

In [48]:
calls_summary = pd.DataFrame({
    "metric": [
        "rows",
        "unique dispatch rows",
        "unique CAD events",
    ],
    "rolling": [
        len(rolling_calls_compare),
        rolling_calls_compare[CALL_KEY].nunique(),
        rolling_calls_compare[CALL_EVENT_KEY].nunique(),
    ],
    "full_refresh": [
        len(full_calls_compare),
        full_calls_compare[CALL_KEY].nunique(),
        full_calls_compare[CALL_EVENT_KEY].nunique(),
    ],
})

calls_summary["difference"] = (
    calls_summary["rolling"]
    - calls_summary["full_refresh"]
)

calls_summary

,metric,rolling,full_refresh,difference
0,rows,1163467,1163468,-1
1,unique dispatch rows,1163467,1163468,-1
2,unique CAD events,681804,681804,0


In [49]:
crime_summary = pd.DataFrame({
    "metric": [
        "rows",
        "unique offenses",
    ],
    "rolling": [
        len(rolling_crime_compare),
        rolling_crime_compare[CRIME_KEY].nunique(),
    ],
    "full_refresh": [
        len(full_crime_compare),
        full_crime_compare[CRIME_KEY].nunique(),
    ],
})

crime_summary["difference"] = (
    crime_summary["rolling"]
    - crime_summary["full_refresh"]
)

crime_summary

,metric,rolling,full_refresh,difference
0,rows,156694,157799,-1105
1,unique offenses,156694,157793,-1099


In [50]:
crime_window_start = rolling_crime[CRIME_EVENT_TIME].min()
crime_window_end = rolling_crime[CRIME_EVENT_TIME].max()

rolling_crime_compare = rolling_crime[
    rolling_crime[CRIME_EVENT_TIME].between(
        crime_window_start,
        crime_window_end,
    )
].copy()

full_crime_compare = full_crime[
    full_crime[CRIME_EVENT_TIME].between(
        crime_window_start,
        crime_window_end,
    )
].copy()

print(
    "Rolling crime window:",
    rolling_crime_compare[CRIME_EVENT_TIME].min(),
    "to",
    rolling_crime_compare[CRIME_EVENT_TIME].max(),
)

print(
    "Full crime window:",
    full_crime_compare[CRIME_EVENT_TIME].min(),
    "to",
    full_crime_compare[CRIME_EVENT_TIME].max(),
)

Rolling crime window: 2024-09-17 23:00:00 to 2026-09-21 22:44:00
Full crime window: 2024-09-17 23:00:00 to 2026-09-21 22:39:00


In [52]:
rolling_crime_daily = (
    rolling_crime_compare
    .assign(
        date=lambda x:
        x[CRIME_EVENT_TIME].dt.normalize()
    )
    .groupby("date")[CRIME_KEY]
    .nunique()
)

full_crime_daily = (
    full_crime_compare
    .assign(
        date=lambda x:
        x[CRIME_EVENT_TIME].dt.normalize()
    )
    .groupby("date")[CRIME_KEY]
    .nunique()
)

crime_daily_compare = pd.concat(
    [
        rolling_crime_daily.rename("rolling"),
        full_crime_daily.rename("full"),
    ],
    axis=1,
).fillna(0)

crime_daily_compare["difference"] = (
    crime_daily_compare["rolling"]
    - crime_daily_compare["full"]
)

crime_daily_compare.head()

,rolling,full,difference
date,,,
2024-09-17,7,7,0
2024-09-18,217,217,0
2024-09-19,231,231,0
2024-09-20,267,267,0
2024-09-21,235,235,0


In [54]:
rolling_crime_ids = set(
    rolling_crime_compare[CRIME_KEY]
    .dropna()
    .astype(str)
)

full_crime_ids = set(
    full_crime_compare[CRIME_KEY]
    .dropna()
    .astype(str)
)

crime_missing_from_rolling = (
    full_crime_ids - rolling_crime_ids
)

crime_extra_in_rolling = (
    rolling_crime_ids - full_crime_ids
)

crime_id_summary = pd.Series({
    "missing_from_rolling":
        len(crime_missing_from_rolling),

    "extra_in_rolling":
        len(crime_extra_in_rolling),
})

crime_id_summary

missing_from_rolling    1103
extra_in_rolling           4
dtype: int64

In [55]:
crime_missing_records = full_crime_compare[
    full_crime_compare[CRIME_KEY]
    .astype(str)
    .isin(crime_missing_from_rolling)
].copy()

crime_missing_records

,report_number,report_date_time,offense_id,offense_date,nibrs_group_a_b,nibrs_crime_against_category,offense_sub_category,shooting_type_group,block_address,latitude,longitude,beat,precinct,sector,neighborhood,reporting_area,offense_category,nibrs_offense_code_description,nibrs_offense_code,census_block_2020
800,2026-282417,2026-09-22 00:00:12,73336213785,2026-09-21 22:39:00,a,person,violation of no contact order,-,REDACTED,NaN,NaN,g1,east,g,first hill,redacted,all other,violation of no contact orders,500,REDACTED
801,2026-282417,2026-09-22 00:00:12,73345345446,2026-09-21 22:39:00,a,person,assault offenses,-,REDACTED,NaN,NaN,g1,east,g,first hill,redacted,all other,simple assault,13B,REDACTED
802,2026-282619,2026-09-22 07:40:41,73345748827,2026-09-21 22:00:00,a,property,motor vehicle theft,-,98XX BLOCK OF 35TH AVE SW,47.515135,-122.376565,w3,southwest,w,roxhill/westwood/arbor heights,5927,property crime,motor vehicle theft,240,12000.1001
803,2026-916524,2026-09-23 13:18:04,73383848346,2026-09-21 22:00:00,a,property,larceny-theft,-,6XX BLOCK OF 36TH AVE,47.606656,-122.287173,g2,east,g,madrona/leschi,8026,property crime,theft of motor vehicle parts or accessories,23G,7800.1028
804,2026-282567,2026-09-23 13:14:39,73452686646,2026-09-21 22:00:00,b,not_a_crime,999,-,REDACTED,NaN,NaN,99,ooj,99,-,redacted,all other,not reportable to nibrs,999,REDACTED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122459,2026-286246,2026-09-25 12:12:26,73493633371,2025-02-25 10:40:00,b,not_a_crime,999,-,12TH AVE S / S JACKSON ST,47.599189,-122.317223,k3,west,k,chinatown/international district,12048,all other,not reportable to nibrs,999,9000.1013
122460,2026-286246,2026-09-25 12:12:26,73537657205,2025-02-25 10:40:00,a,society,narcotic violations (includes drug equip.),-,12TH AVE S / S JACKSON ST,47.599189,-122.317223,k3,west,k,chinatown/international district,12048,all other,drug equipment violations,35B,9000.1013
123484,2025-047367,2025-02-20 16:51:42,62859441899,2025-02-20 12:00:00,a,property,motor vehicle theft,-,22XX BLOCK OF E MADISON ST,47.618795,-122.303099,c3,east,c,central area/squire park,1906,property crime,motor vehicle theft,240,7902.2
130525,2025-013241,2025-01-15 07:34:57,73245471321,2025-01-15 06:05:00,a,person,assault offenses,-,14XX BLOCK OF BOREN AVE,47.612969,-122.328459,e3,east,e,first hill,4211,all other,simple assault,13B,-


In [56]:
crime_extra_records = rolling_crime_compare[
    rolling_crime_compare[CRIME_KEY]
    .astype(str)
    .isin(crime_extra_in_rolling)
].copy()

crime_extra_records

,report_number,report_date_time,offense_id,offense_date,nibrs_group_a_b,nibrs_crime_against_category,offense_sub_category,shooting_type_group,block_address,latitude,longitude,beat,precinct,sector,neighborhood,reporting_area,offense_category,nibrs_offense_code_description,nibrs_offense_code,census_block_2020
153336,2026-257025,2026-08-29 21:46:23,72513757285,2026-08-29 19:28:00,a,property,larceny-theft,shots fired (eyewitness/casings/property damage),REDACTED,NaN,NaN,s2,south,s,rainier beach,redacted,property crime,theft from motor vehicle,23F,REDACTED
153339,2026-257025,2026-08-29 21:46:23,72513345011,2026-08-29 19:28:00,a,property,robbery,shots fired (eyewitness/casings/property damage),REDACTED,NaN,NaN,s2,south,s,rainier beach,redacted,violent crime,robbery,120,REDACTED
155382,2026-270868,2026-09-11 11:27:41,73063212435,2026-09-11 08:41:00,-,-,unknown,shooting (non-fatal injury),32XX BLOCK OF FUHRMAN AVE E,47.650383,-122.320197,c1,east,c,montlake/portage bay,1932,all other,-,-,6100.1012
155972,2026-275787,2026-09-15 22:34:54,73235511023,2026-09-15 20:05:00,-,-,unknown,shots fired (eyewitness/casings/property damage),CAROLINA ST / OREGON AVE,47.654431,-122.413180,q1,west,q,magnolia,7363,all other,-,-,-


In [60]:
print(
    "Rolling duplicate offense_ids:",
    rolling_crime_compare[CRIME_KEY].duplicated().sum()
)

print(
    "Full duplicate offense_ids:",
    full_crime_compare[CRIME_KEY].duplicated().sum()
)

print(
    "Rolling unique IDs:",
    rolling_crime_compare[CRIME_KEY].nunique(),
    "of",
    len(rolling_crime_compare),
)

print(
    "Full unique IDs:",
    full_crime_compare[CRIME_KEY].nunique(),
    "of",
    len(full_crime_compare),
)

Rolling duplicate offense_ids: 0
Full duplicate offense_ids: 6
Rolling unique IDs: 156694 of 156694
Full unique IDs: 157793 of 157799


In [61]:
def compare_record_fields(
    rolling_df,
    full_df,
    key,
    ignore_columns=None,
):
    ignore_columns = set(ignore_columns or [])

    common_columns = sorted(
        (
            set(rolling_df.columns)
            & set(full_df.columns)
        )
        - {key}
        - ignore_columns
    )

    # We expect one row per key for record-level comparison.
    # Fail explicitly if that assumption isn't true.
    rolling_duplicates = (
        rolling_df[key]
        .dropna()
        .duplicated()
        .sum()
    )

    full_duplicates = (
        full_df[key]
        .dropna()
        .duplicated()
        .sum()
    )

    if rolling_duplicates:
        raise ValueError(
            f"Rolling dataframe contains "
            f"{rolling_duplicates} duplicate {key} values."
        )

    if full_duplicates:
        raise ValueError(
            f"Full dataframe contains "
            f"{full_duplicates} duplicate {key} values."
        )

    merged = rolling_df[
        [key] + common_columns
    ].merge(
        full_df[
            [key] + common_columns
        ],
        on=key,
        how="inner",
        suffixes=("_rolling", "_full"),
        validate="one_to_one",
    )

    differences = []

    for column in common_columns:
        rolling_col = f"{column}_rolling"
        full_col = f"{column}_full"

        rolling_values = (
            merged[rolling_col]
            .astype("string")
            .fillna("<NA>")
        )

        full_values = (
            merged[full_col]
            .astype("string")
            .fillna("<NA>")
        )

        changed = rolling_values.ne(full_values)

        if changed.any():
            part = pd.DataFrame({
                key: merged.loc[changed, key].values,
                "column": column,
                "rolling_value":
                    merged.loc[
                        changed,
                        rolling_col,
                    ].values,
                "full_value":
                    merged.loc[
                        changed,
                        full_col,
                    ].values,
            })

            differences.append(part)

    if not differences:
        return pd.DataFrame(
            columns=[
                key,
                "column",
                "rolling_value",
                "full_value",
            ]
        )

    return pd.concat(
        differences,
        ignore_index=True,
    )

In [62]:
crime_field_diffs = compare_record_fields(
    rolling_crime_compare,
    full_crime_compare,
    CRIME_KEY,
)

crime_field_diffs.head()

ValueError: Full dataframe contains 6 duplicate offense_id values.

In [30]:
rolling_daily = (
    rolling_calls_compare
    .assign(date=lambda x: x[CALL_TIME].dt.date)
    .groupby("date")[CALL_EVENT_KEY]
    .nunique()
)

full_daily = (
    full_calls_compare
    .assign(date=lambda x: x[CALL_TIME].dt.date)
    .groupby("date")[CALL_EVENT_KEY]
    .nunique()
)

daily_compare = pd.concat(
    [
        rolling_daily.rename("rolling"),
        full_daily.rename("full"),
    ],
    axis=1,
).fillna(0)

daily_compare["difference"] = (
    daily_compare["rolling"] - daily_compare["full"]
)

In [31]:
daily_compare[daily_compare["difference"] != 0]

,rolling,full,difference
date,,,


In [32]:
rolling_ids = set(
    rolling_calls_compare[CALL_KEY].dropna()
)

full_ids = set(
    full_calls_compare[CALL_KEY].dropna()
)

missing_from_rolling = full_ids - rolling_ids
extra_in_rolling = rolling_ids - full_ids

In [33]:
pd.Series({
    "missing_from_rolling": len(missing_from_rolling),
    "extra_in_rolling": len(extra_in_rolling),
})

missing_from_rolling    1
extra_in_rolling        0
dtype: int64

In [34]:
full_calls_compare[
    full_calls_compare[CALL_KEY].isin(missing_from_rolling)
]

,cad_event_number,cad_event_original_time_queued,cad_event_arrived_time,cad_event_clearance_description,call_sign_dispatch_id,call_type,priority,initial_call_type,final_call_type,cad_event_response_category,dispatch_precinct,dispatch_sector,dispatch_beat,dispatch_neighborhood,dispatch_latitude,dispatch_longitude,count_of_officers,event_group
14294,2026000278948,2026-09-18 15:55:36,2026-09-18 18:23:53,unable to locate incident or complainant,20260002789482C3,911,3,threats (incls in-person/by phone/in writing),suspicious circum. - suspicious person,spd,east,edward,e2,capitol hill,47.613496,-122.316785,2,suspicious circumstances


In [35]:
rolling_calls_compare[
    rolling_calls_compare[CALL_KEY].isin(extra_in_rolling)
]

,cad_event_number,cad_event_original_time_queued,cad_event_arrived_time,cad_event_clearance_description,call_sign_dispatch_id,call_type,priority,initial_call_type,final_call_type,cad_event_response_category,dispatch_precinct,dispatch_sector,dispatch_beat,dispatch_neighborhood,dispatch_latitude,dispatch_longitude,count_of_officers,event_group


In [36]:
joined = rolling_calls_compare.merge(
    full_calls_compare,
    on=CALL_KEY,
    how="inner",
    suffixes=("_rolling", "_full"),
)

In [37]:
def compare_record_fields(
    rolling_df,
    full_df,
    key,
    ignore_columns=None,
):
    ignore_columns = set(ignore_columns or [])

    common_columns = (
        set(rolling_df.columns)
        & set(full_df.columns)
    )

    compare_columns = sorted(
        common_columns - {key} - ignore_columns
    )

    left = rolling_df.set_index(key)
    right = full_df.set_index(key)

    common_ids = left.index.intersection(right.index)

    left = left.loc[common_ids, compare_columns]
    right = right.loc[common_ids, compare_columns]

    differences = []

    for column in compare_columns:
        a = left[column].astype("string").fillna("<NA>")
        b = right[column].astype("string").fillna("<NA>")

        changed = a != b

        if changed.any():
            part = pd.DataFrame({
                key: common_ids[changed],
                "column": column,
                "rolling_value": a[changed].values,
                "full_value": b[changed].values,
            })
            differences.append(part)

    if not differences:
        return pd.DataFrame(
            columns=[
                key,
                "column",
                "rolling_value",
                "full_value",
            ]
        )

    return pd.concat(differences, ignore_index=True)

In [38]:
call_field_diffs = compare_record_fields(
    rolling_calls_compare,
    full_calls_compare,
    CALL_KEY,
)

In [39]:
call_field_diffs[CALL_KEY].nunique()

20

In [40]:
(
    call_field_diffs
    .groupby("column")
    .size()
    .sort_values(ascending=False)
)

column
final_call_type       20
event_group            8
dispatch_longitude     5
dispatch_latitude      5
dtype: int64

In [42]:
changed_ids = set(
    call_field_diffs[CALL_KEY]
)

changed_calls = full_calls_compare[
    full_calls_compare[CALL_KEY].isin(changed_ids)
].copy()

latest_call_time = full_calls_compare[CALL_TIME].max()

changed_calls["age_days"] = (
    latest_call_time
    - changed_calls[CALL_TIME]
).dt.days

In [43]:
changed_calls["age_days"].describe(
    percentiles=[.5, .75, .9, .95, .99]
)

count    20.000000
mean      3.050000
std       0.759155
min       2.000000
50%       3.000000
75%       4.000000
90%       4.000000
95%       4.000000
99%       4.000000
max       4.000000
Name: age_days, dtype: float64

In [63]:
import pandas as pd
import plotly.graph_objects as go

from dashboard.crime_dashboard_data import (
    prepare_crime_snapshot,
    EVENT_ID_COLUMN,
    TIME_COLUMN,
)

from dashboard.crime_dashboard_figures import (
    prepare_daily_event_data,
    TARGET_CRIME_CATEGORIES,
)


# ---------------------------------------------------------
# Build dashboard-style contexts from BOTH source DataFrames
# ---------------------------------------------------------

def make_daily_context(source_df):
    """
    Reproduce the part of load_crime_dashboard_context()
    used by the dashboard daily time-series.
    """
    prepared = prepare_crime_snapshot(source_df.copy())

    analysis_df = prepared.loc[
        ~prepared["is_excluded_from_crime_analysis"]
    ].copy()

    valid_time = analysis_df[
        analysis_df[TIME_COLUMN].notna()
        & analysis_df[EVENT_ID_COLUMN].notna()
    ].copy()

    return {
        "valid_time": valid_time,
    }


rolling_context = make_daily_context(
    rolling_crime_compare
)

full_context = make_daily_context(
    full_crime_compare
)


# ---------------------------------------------------------
# Run the EXACT dashboard daily-series preparation function
# ---------------------------------------------------------

rolling_daily, rolling_window = prepare_daily_event_data(
    context=rolling_context,
    selected_bins=TARGET_CRIME_CATEGORIES,
    analysis_state=None,
)

full_daily, full_window = prepare_daily_event_data(
    context=full_context,
    selected_bins=TARGET_CRIME_CATEGORIES,
    analysis_state=None,
)


# ---------------------------------------------------------
# Align both outputs to the same dates for QA comparison
# ---------------------------------------------------------

comparison = (
    rolling_daily[
        [
            "date",
            "reported_offenses",
            "rolling_7_day_avg",
        ]
    ]
    .rename(
        columns={
            "reported_offenses":
                "rolling_reported_offenses",
            "rolling_7_day_avg":
                "rolling_7_day_avg",
        }
    )
    .merge(
        full_daily[
            [
                "date",
                "reported_offenses",
                "rolling_7_day_avg",
            ]
        ].rename(
            columns={
                "reported_offenses":
                    "full_reported_offenses",
                "rolling_7_day_avg":
                    "full_7_day_avg",
            }
        ),
        on="date",
        how="inner",
        validate="one_to_one",
    )
)

comparison["difference"] = (
    comparison["rolling_reported_offenses"]
    - comparison["full_reported_offenses"]
)


# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

fig = go.Figure()

# Rolling snapshot -- daily
fig.add_trace(
    go.Scatter(
        x=comparison["date"],
        y=comparison["rolling_reported_offenses"],
        mode="lines",
        name="Rolling snapshot — daily",
        line=dict(width=1.4),
        opacity=0.45,
        hovertemplate=(
            "<b>%{x|%Y-%m-%d}</b><br>"
            "Rolling daily offenses: %{y:,}"
            "<extra></extra>"
        ),
    )
)

# Full refresh -- daily
fig.add_trace(
    go.Scatter(
        x=comparison["date"],
        y=comparison["full_reported_offenses"],
        mode="lines",
        name="Full refresh — daily",
        line=dict(width=1.4, dash="dot"),
        opacity=0.60,
        hovertemplate=(
            "<b>%{x|%Y-%m-%d}</b><br>"
            "Full-refresh daily offenses: %{y:,}"
            "<extra></extra>"
        ),
    )
)

# Rolling snapshot -- 7-day average
fig.add_trace(
    go.Scatter(
        x=comparison["date"],
        y=comparison["rolling_7_day_avg"],
        mode="lines",
        name="Rolling snapshot — 7-day average",
        line=dict(width=3),
        hovertemplate=(
            "<b>%{x|%Y-%m-%d}</b><br>"
            "Rolling 7-day average: %{y:,.1f}"
            "<extra></extra>"
        ),
    )
)

# Full refresh -- 7-day average
fig.add_trace(
    go.Scatter(
        x=comparison["date"],
        y=comparison["full_7_day_avg"],
        mode="lines",
        name="Full refresh — 7-day average",
        line=dict(width=3, dash="dot"),
        hovertemplate=(
            "<b>%{x|%Y-%m-%d}</b><br>"
            "Full-refresh 7-day average: %{y:,.1f}"
            "<extra></extra>"
        ),
    )
)


# ---------------------------------------------------------
# Match dashboard-style controls/layout
# ---------------------------------------------------------

plot_start = comparison["date"].min()
plot_end = comparison["date"].max()

fig.update_layout(
    title=dict(
        text=(
            "Daily Crime Events — Rolling Snapshot vs Full Refresh"
            "<br><sup>Using dashboard crime-series preparation code</sup>"
        ),
        x=0.01,
        xanchor="left",
    ),
    hovermode="x unified",
    xaxis=dict(
        title=None,
        range=[
            max(
                plot_start,
                plot_end - pd.Timedelta(days=1),
            ),
            plot_end,
        ],
        rangeselector=dict(
            buttons=[
                dict(
                    count=1,
                    label="1D",
                    step="day",
                    stepmode="backward",
                ),
                dict(
                    count=7,
                    label="1W",
                    step="day",
                    stepmode="backward",
                ),
                dict(
                    count=1,
                    label="1M",
                    step="month",
                    stepmode="backward",
                ),
                dict(
                    count=1,
                    label="1Y",
                    step="year",
                    stepmode="backward",
                ),
            ],
        ),
        rangeslider=dict(
            visible=True,
            range=[plot_start, plot_end],
        ),
    ),
    yaxis=dict(
        title="Reported offenses",
    ),
    legend_title_text="Source / metric",
)

fig.show()


# ---------------------------------------------------------
# QA summary
# ---------------------------------------------------------

different_days = comparison[
    comparison["difference"] != 0
]

print(
    f"Days compared: {len(comparison):,}"
)

print(
    f"Days with different dashboard crime totals: "
    f"{len(different_days):,}"
)

print(
    f"Maximum absolute daily discrepancy: "
    f"{comparison['difference'].abs().max():,}"
)

display(
    different_days[
        [
            "date",
            "rolling_reported_offenses",
            "full_reported_offenses",
            "difference",
        ]
    ]
)

Days compared: 366
Days with different dashboard crime totals: 108
Maximum absolute daily discrepancy: 70


,date,rolling_reported_offenses,full_reported_offenses,difference
10,2025-10-01,256,257,-1
24,2025-10-15,222,223,-1
29,2025-10-20,220,221,-1
54,2025-11-14,209,210,-1
65,2025-11-25,200,201,-1
...,...,...,...,...
361,2026-09-17,118,162,-44
362,2026-09-18,108,156,-48
363,2026-09-19,100,144,-44
364,2026-09-20,101,124,-23


In [65]:
import pandas as pd

# ---------------------------------------------------------
# Basic coverage comparison
# ---------------------------------------------------------

for label, df in [
    ("ROLLING", rolling_crime_compare),
    ("FULL", full_crime_compare),
]:
    print(f"\n{label}")
    print("-" * 50)

    print("Rows:", f"{len(df):,}")
    print("Unique offenses:", f"{df['offense_id'].nunique():,}")

    offense_dates = pd.to_datetime(
        df["offense_date"],
        errors="coerce",
    )

    report_dates = pd.to_datetime(
        df["report_date_time"],
        errors="coerce",
    )

    print("Earliest offense_date:", offense_dates.min())
    print("Latest offense_date:", offense_dates.max())

    print("Earliest report_date_time:", report_dates.min())
    print("Latest report_date_time:", report_dates.max())


# ---------------------------------------------------------
# Daily raw offense totals
# ---------------------------------------------------------

rolling_recent = (
    rolling_crime_compare
    .assign(
        offense_day=pd.to_datetime(
            rolling_crime_compare["offense_date"],
            errors="coerce",
        ).dt.normalize()
    )
    .groupby("offense_day")["offense_id"]
    .nunique()
    .rename("rolling")
)

full_recent = (
    full_crime_compare
    .assign(
        offense_day=pd.to_datetime(
            full_crime_compare["offense_date"],
            errors="coerce",
        ).dt.normalize()
    )
    .groupby("offense_day")["offense_id"]
    .nunique()
    .rename("full")
)

recent_check = pd.concat(
    [rolling_recent, full_recent],
    axis=1,
).fillna(0)

recent_check["difference"] = (
    recent_check["rolling"]
    - recent_check["full"]
)

recent_check["missing_from_rolling"] = (
    recent_check["full"]
    - recent_check["rolling"]
)

display(recent_check.tail(60))


# ---------------------------------------------------------
# IDs present in full pull but absent from rolling
# ---------------------------------------------------------

rolling_ids = set(
    rolling_crime_compare["offense_id"]
    .dropna()
    .astype(str)
)

full_ids = set(
    full_crime_compare["offense_id"]
    .dropna()
    .astype(str)
)

missing_ids = full_ids - rolling_ids

missing_crimes = (
    full_crime_compare[
        full_crime_compare["offense_id"]
        .astype(str)
        .isin(missing_ids)
    ]
    .copy()
)

missing_crimes["offense_date"] = pd.to_datetime(
    missing_crimes["offense_date"],
    errors="coerce",
)

missing_crimes["report_date_time"] = pd.to_datetime(
    missing_crimes["report_date_time"],
    errors="coerce",
)

print(
    "\nOffenses in FULL but absent from ROLLING:",
    f"{len(missing_crimes):,}",
)

display(
    missing_crimes[
        [
            "offense_id",
            "offense_date",
            "report_date_time",
            "offense_category",
            "offense_sub_category",
        ]
    ]
    .sort_values("offense_date")
    .tail(100)
)


ROLLING
--------------------------------------------------
Rows: 156,694
Unique offenses: 156,694
Earliest offense_date: 2024-09-17 23:00:00
Latest offense_date: 2026-09-21 22:44:00
Earliest report_date_time: 2023-11-09 06:40:36
Latest report_date_time: 2026-09-21 23:58:02

FULL
--------------------------------------------------
Rows: 157,799
Unique offenses: 157,793
Earliest offense_date: 2024-09-17 23:00:00
Latest offense_date: 2026-09-21 22:39:00
Earliest report_date_time: 2023-11-09 06:40:36
Latest report_date_time: 2026-09-28 16:35:56


,rolling,full,difference,missing_from_rolling
offense_day,,,,
2026-07-24,195,196,-1,1
2026-07-25,203,204,-1,1
2026-07-26,202,206,-4,4
2026-07-27,187,188,-1,1
2026-07-28,215,216,-1,1
2026-07-29,216,217,-1,1
2026-07-30,218,219,-1,1
2026-07-31,237,239,-2,2
2026-08-01,239,247,-8,8



Offenses in FULL but absent from ROLLING: 1,103


,offense_id,offense_date,report_date_time,offense_category,offense_sub_category
1021,73319640930,2026-09-20 15:40:00,2026-09-20 19:04:43,all other,"property offenses (includes stolen, destruction)"
1012,73337569063,2026-09-20 16:30:00,2026-09-22 02:17:19,all other,assault offenses
1007,73352476753,2026-09-20 17:00:00,2026-09-22 17:27:52,property crime,motor vehicle theft
1006,73335113759,2026-09-20 17:30:00,2026-09-21 20:08:49,property crime,larceny-theft
1004,73543362928,2026-09-20 18:00:00,2026-09-26 11:08:25,property crime,larceny-theft
...,...,...,...,...,...
804,73452686646,2026-09-21 22:00:00,2026-09-23 13:14:39,all other,999
803,73383848346,2026-09-21 22:00:00,2026-09-23 13:18:04,property crime,larceny-theft
802,73345748827,2026-09-21 22:00:00,2026-09-22 07:40:41,property crime,motor vehicle theft
801,73345345446,2026-09-21 22:39:00,2026-09-22 00:00:12,all other,assault offenses


In [66]:
import pandas as pd

rolling_offense_max = pd.to_datetime(
    rolling_crime["offense_date"],
    errors="coerce"
).max()

full_offense_max = pd.to_datetime(
    full_crime["offense_date"],
    errors="coerce"
).max()

rolling_report_max = pd.to_datetime(
    rolling_crime["report_date_time"],
    errors="coerce"
).max()

full_report_max = pd.to_datetime(
    full_crime["report_date_time"],
    errors="coerce"
).max()

pd.DataFrame({
    "rolling": [
        rolling_offense_max,
        rolling_report_max,
    ],
    "full_current_source": [
        full_offense_max,
        full_report_max,
    ],
}, index=[
    "latest offense_date",
    "latest report_date_time",
])

,rolling,full_current_source
latest offense_date,2026-09-21 22:44:00,2026-09-28 20:26:00
latest report_date_time,2026-09-21 23:58:02,2026-09-28 22:31:29


In [67]:
rolling_latest_offense_day = (
    pd.to_datetime(
        rolling_crime["offense_date"],
        errors="coerce"
    )
    .max()
    .normalize()
)

source_latest_offense_day = (
    pd.to_datetime(
        full_crime["offense_date"],
        errors="coerce"
    )
    .max()
    .normalize()
)

print("Current freshness logic:")
print("Rolling:", rolling_latest_offense_day)
print("Source: ", source_latest_offense_day)
print(
    "Would pass:",
    rolling_latest_offense_day
    == source_latest_offense_day
)

Current freshness logic:
Rolling: 2026-09-21 00:00:00
Source:  2026-09-28 00:00:00
Would pass: False


In [68]:
rolling_latest_report_day = (
    pd.to_datetime(
        rolling_crime["report_date_time"],
        errors="coerce"
    )
    .max()
    .normalize()
)

source_latest_report_day = (
    pd.to_datetime(
        full_crime["report_date_time"],
        errors="coerce"
    )
    .max()
    .normalize()
)

print("\nReport-date freshness logic:")
print("Rolling:", rolling_latest_report_day)
print("Source: ", source_latest_report_day)
print(
    "Would pass:",
    rolling_latest_report_day
    == source_latest_report_day
)


Report-date freshness logic:
Rolling: 2026-09-21 00:00:00
Source:  2026-09-28 00:00:00
Would pass: False


In [69]:
cutoff = pd.to_datetime(
    rolling_crime["report_date_time"],
    errors="coerce"
).max()

missing_after_cutoff = missing_crimes[
    missing_crimes["report_date_time"] > cutoff
].copy()

print("Total missing offenses:", len(missing_crimes))

print(
    "Missing offenses reported after rolling cutoff:",
    len(missing_after_cutoff)
)

print(
    "Percent explained by later reporting:",
    f"{len(missing_after_cutoff) / len(missing_crimes) * 100:.2f}%"
)

Total missing offenses: 1103
Missing offenses reported after rolling cutoff: 959
Percent explained by later reporting: 86.94%


In [70]:
cutoff = pd.to_datetime(
    rolling_crime["report_date_time"],
    errors="coerce"
).max()

unexplained_missing = missing_crimes[
    missing_crimes["report_date_time"] <= cutoff
].copy()

print("Missing offenses not explained by later reporting:",
      len(unexplained_missing))

display(
    unexplained_missing[
        [
            "offense_id",
            "offense_date",
            "report_date_time",
            "offense_category",
            "offense_sub_category",
        ]
    ]
    .sort_values("report_date_time")
)

Missing offenses not explained by later reporting: 144


,offense_id,offense_date,report_date_time,offense_category,offense_sub_category
130525,73245471321,2025-01-15 06:05:00,2025-01-15 07:34:57,all other,assault offenses
123484,62859441899,2025-02-20 12:00:00,2025-02-20 16:51:42,property crime,motor vehicle theft
104983,64094797193,2025-05-22 11:00:00,2025-05-22 14:56:32,property crime,larceny-theft
104031,64141176153,2025-05-26 17:48:00,2025-05-26 19:48:32,all other,assault offenses
90458,65447871758,2025-07-29 00:00:00,2025-08-13 17:40:09,all other,999
...,...,...,...,...,...
2149,73337624727,2026-09-14 00:00:00,2026-09-21 22:45:50,all other,999
3133,73335896378,2026-09-08 19:49:00,2026-09-21 22:45:50,all other,999
840,73335982695,2026-09-21 16:40:00,2026-09-21 23:02:07,all other,999
841,73336056881,2026-09-21 16:25:00,2026-09-21 23:22:28,violent crime,robbery


In [71]:
last_refresh_report_day = pd.to_datetime(
    rolling_crime["report_date_time"],
    errors="coerce"
).max().normalize()

overlap_start = (
    last_refresh_report_day
    - pd.Timedelta(days=30)
)

unexplained_missing["inside_30_day_overlap"] = (
    unexplained_missing["report_date_time"] >= overlap_start
)

unexplained_missing["inside_30_day_overlap"].value_counts()

inside_30_day_overlap
True     125
False     19
Name: count, dtype: int64

In [72]:
from dashboard.crime_service import load_crime_dataset
import pandas as pd

rolling_report_max = pd.to_datetime(
    rolling_crime["report_date_time"],
    errors="coerce"
).max()

rolling_fetch_start = (
    rolling_report_max.normalize()
    - pd.Timedelta(days=30)
).date().isoformat()

print("Rolling-style fetch start:", rolling_fetch_start)

incremental_control = load_crime_dataset(
    start_date=rolling_fetch_start,
    page_size=5000,
    max_pages=None,
    date_column="report_date_time",
)

incremental_control_ids = set(
    incremental_control["offense_id"]
    .dropna()
    .astype(str)
)

inside_overlap_ids = set(
    unexplained_missing.loc[
        unexplained_missing["inside_30_day_overlap"],
        "offense_id"
    ]
    .astype(str)
)

found_now = inside_overlap_ids & incremental_control_ids
still_not_returned = inside_overlap_ids - incremental_control_ids

print("Expected missing IDs inside overlap:", len(inside_overlap_ids))
print("Returned by rolling-style query now:", len(found_now))
print("Still not returned:", len(still_not_returned))

Rolling-style fetch start: 2026-08-22
Fetched page 1, records this page: 5000, total records: 5000
Fetched page 2, records this page: 3488, total records: 8488
Expected missing IDs inside overlap: 125
Returned by rolling-style query now: 125
Still not returned: 0


In [73]:
import pandas as pd

sim_existing = rolling_crime.copy()
sim_new = incremental_control.copy()

sim_existing["offense_date"] = pd.to_datetime(
    sim_existing["offense_date"],
    errors="coerce",
)
sim_existing["report_date_time"] = pd.to_datetime(
    sim_existing["report_date_time"],
    errors="coerce",
)

sim_new["offense_date"] = pd.to_datetime(
    sim_new["offense_date"],
    errors="coerce",
)
sim_new["report_date_time"] = pd.to_datetime(
    sim_new["report_date_time"],
    errors="coerce",
)

sim_combined = pd.concat(
    [sim_existing, sim_new],
    ignore_index=True,
)

sim_combined = sim_combined.drop_duplicates(
    subset=["offense_id"],
    keep="last",
)

latest_offense = sim_combined["offense_date"].max()

cutoff = (
    latest_offense
    - pd.Timedelta(days=734)
)

sim_combined = sim_combined[
    sim_combined["offense_date"] >= cutoff
].copy()

sim_ids = set(
    sim_combined["offense_id"]
    .dropna()
    .astype(str)
)

missing_after_simulated_refresh = (
    full_ids - sim_ids
)

print(
    "Missing from rolling before simulation:",
    len(full_ids - rolling_ids),
)

print(
    "Missing after simulated incremental refresh:",
    len(missing_after_simulated_refresh),
)

Missing from rolling before simulation: 1103
Missing after simulated incremental refresh: 1678


In [74]:
# The simulated rolling snapshot has already been refreshed and trimmed.
sim_latest = sim_combined["offense_date"].max()
sim_cutoff = sim_latest - pd.Timedelta(days=734)

# Put the authoritative full pull into THE SAME rolling window.
full_control = full_crime.copy()

full_control["offense_date"] = pd.to_datetime(
    full_control["offense_date"],
    errors="coerce",
)

full_control = full_control[
    full_control["offense_date"].between(
        sim_cutoff,
        sim_latest,
    )
].copy()

sim_control = sim_combined[
    sim_combined["offense_date"].between(
        sim_cutoff,
        sim_latest,
    )
].copy()

full_control_ids = set(
    full_control["offense_id"]
    .dropna()
    .astype(str)
)

sim_control_ids = set(
    sim_control["offense_id"]
    .dropna()
    .astype(str)
)

missing_after_refresh = (
    full_control_ids - sim_control_ids
)

extra_after_refresh = (
    sim_control_ids - full_control_ids
)

print("Comparison window:")
print("Start:", sim_cutoff)
print("End:  ", sim_latest)

print("\nFull unique offenses:",
      len(full_control_ids))

print("Simulated rolling unique offenses:",
      len(sim_control_ids))

print("\nMissing after simulated refresh:",
      len(missing_after_refresh))

print("Extra after simulated refresh:",
      len(extra_after_refresh))

Comparison window:
Start: 2024-09-24 20:26:00
End:   2026-09-28 20:26:00

Full unique offenses: 156934
Simulated rolling unique offenses: 156919

Missing after simulated refresh: 19
Extra after simulated refresh: 4


In [75]:
# ---------------------------------------------------------
# Inspect residual discrepancies after simulated refresh
# ---------------------------------------------------------

missing_residual = full_control[
    full_control["offense_id"]
    .astype(str)
    .isin(missing_after_refresh)
].copy()

extra_residual = sim_control[
    sim_control["offense_id"]
    .astype(str)
    .isin(extra_after_refresh)
].copy()

columns = [
    "offense_id",
    "offense_date",
    "report_date_time",
    "report_number",
    "offense_category",
    "offense_sub_category",
]

print("MISSING FROM SIMULATED ROLLING")
print("=" * 60)

display(
    missing_residual[
        [c for c in columns if c in missing_residual.columns]
    ]
    .sort_values("report_date_time")
)

print("\nEXTRA IN SIMULATED ROLLING")
print("=" * 60)

display(
    extra_residual[
        [c for c in columns if c in extra_residual.columns]
    ]
    .sort_values("report_date_time")
)

MISSING FROM SIMULATED ROLLING


,offense_id,offense_date,report_date_time,report_number,offense_category,offense_sub_category
130525,73245471321,2025-01-15 06:05:00,2025-01-15 07:34:57,2025-013241,all other,assault offenses
123484,62859441899,2025-02-20 12:00:00,2025-02-20 16:51:42,2025-047367,property crime,motor vehicle theft
104983,64094797193,2025-05-22 11:00:00,2025-05-22 14:56:32,2025-138142,property crime,larceny-theft
104031,64141176153,2025-05-26 17:48:00,2025-05-26 19:48:32,2025-142668,all other,assault offenses
90458,65447871758,2025-07-29 00:00:00,2025-08-13 17:40:09,2025-232378,all other,999
72671,73361150674,2025-10-15 08:00:00,2025-10-16 19:32:21,2025-302368,all other,"property offenses (includes stolen, destruction)"
52639,68417281319,2026-01-14 13:20:00,2026-02-04 17:24:55,2026-034177,all other,extortion/fraud/forgery/bribery (includes bad ...
46518,68551896061,2026-02-13 16:00:00,2026-02-13 20:16:27,2026-043093,all other,extortion/fraud/forgery/bribery (includes bad ...
65849,72591863682,2025-11-14 00:00:00,2026-03-09 09:19:17,2026-028763,all other,pornography
65850,72591924528,2025-11-14 00:00:00,2026-03-09 09:19:17,2026-028763,all other,999



EXTRA IN SIMULATED ROLLING


,offense_id,offense_date,report_date_time,report_number,offense_category,offense_sub_category
153336,72513757285,2026-08-29 19:28:00,2026-08-29 21:46:23,2026-257025,property crime,larceny-theft
153339,72513345011,2026-08-29 19:28:00,2026-08-29 21:46:23,2026-257025,violent crime,robbery
155382,73063212435,2026-09-11 08:41:00,2026-09-11 11:27:41,2026-270868,all other,unknown
155972,73235511023,2026-09-15 20:05:00,2026-09-15 22:34:54,2026-275787,all other,unknown


In [76]:
fetch_start = pd.Timestamp(rolling_fetch_start)

missing_residual["inside_incremental_fetch_window"] = (
    missing_residual["report_date_time"]
    >= fetch_start
)

print(
    missing_residual[
        "inside_incremental_fetch_window"
    ].value_counts()
)

inside_incremental_fetch_window
False    19
Name: count, dtype: int64


In [77]:
extra_ids = set(extra_after_refresh)

# Are these IDs anywhere in today's full source pull,
# even outside the current comparison window?
extra_in_full_anywhere = full_crime[
    full_crime["offense_id"]
    .astype(str)
    .isin(extra_ids)
].copy()

print("Extra IDs:", len(extra_ids))
print(
    "Extra IDs still present anywhere in current full pull:",
    extra_in_full_anywhere["offense_id"].nunique()
)

if not extra_in_full_anywhere.empty:
    display(
        extra_in_full_anywhere[
            [
                "offense_id",
                "offense_date",
                "report_date_time",
                "report_number",
                "offense_category",
                "offense_sub_category",
            ]
        ].sort_values("offense_date")
    )

print("\nIDs absent from current source entirely:")
print(
    sorted(
        extra_ids
        - set(
            extra_in_full_anywhere["offense_id"]
            .astype(str)
        )
    )
)

Extra IDs: 4
Extra IDs still present anywhere in current full pull: 0

IDs absent from current source entirely:
['72513345011', '72513757285', '73063212435', '73235511023']
